In [9]:
import numpy as np
from ortools.linear_solver import pywraplp
from ortools.sat.python import cp_model
import random

In [ ]:
with open("../../data/input.txt") as f:
# with open("./data/test.txt") as f:
    n, m, bat_cap, T = [int(i) for i in f.readline().strip().split()] # |V|, |E|, S, T

    build_cost, recharge, profit = [int(i) for i in f.readline().strip().split()] # C, E, P

    # edge_list = [tuple([int(i) for i in f.readline().strip().split()]) for _ in range(m)] # E
    
    dist = np.ndarray((n, n), dtype=np.float32)

    for i in range(m):
        u, v, cost = f.readline().strip().split()
        
        u = int(u) - 1
        v = int(v) - 1
        cost = float(cost)

        dist[u, v] = cost
        dist[v, u] = cost
        dist[u, u] = 0


    demand = [int(i) for i in f.readline().strip().split()] # D



print(n, m, bat_cap, T)
print(build_cost, recharge, profit)
print(dist)
print(demand)

In [ ]:
# Hyperparameters
alpha = 1
beta = 0
x = [1 for _ in range(n)]
print(x)

In [ ]:
solver = pywraplp.Solver.CreateSolver("SCIP")
inf = solver.infinity()
f = [[[solver.IntVar(0, inf, f"f_{t}_{i}_{j}") for j in range(n)] for i in range(n)] for t in range(T)]
I = [[solver.IntVar(0, bat_cap, f"I_{t}_{j}") for j in range(n)] for t in range(T)]

for t in range(T):
    for j in range(n):
        solver.Add(solver.Sum(f[t][i][j] for i in range(n)) <= I[t][j])

for t in range(1, T):
    for j in range(n):
        solver.Add(I[t][j] == I[t - 1][j] - solver.Sum(f[t - 1][i][j] for i in range(n)) + recharge)

for j in range(n):
    solver.Add(I[0][j] == bat_cap)

for t in range(T):
    for i in range(n):
        solver.Add(solver.Sum(f[t][i][j] for j in range(n)) <= demand[i])
        for j in range(n):
            solver.Add(f[t][i][j] <= x[j] * demand[i]) 

solver.Minimize(solver.Sum([alpha * (demand[i] - solver.Sum(f[t][i][j] for j in range(n))) 
                           + beta * solver.Sum(f[t][i][j] * dist[i][j] for j in range(n)) 
                           for i in range(n) for t in range(T)]))

In [ ]:
print(solver.NumVariables())
print(solver.NumConstraints())

In [ ]:
print(f"Solving with {solver.SolverVersion()}")
solver.EnableOutput()
solver.SetTimeLimit(60000)
status = solver.Solve()
status

In [ ]:
if status == pywraplp.Solver.OPTIMAL:
    print("Objective value = ", solver.Objective().Value())
elif status == pywraplp.Solver.INFEASIBLE:
    print("No solution!")

---

$$
\begin{align*}
    \argmin_{f \in \mathbb{N}^{{\left|V\right|^2}T}}&\sum_{t = 0}^{T - 1}\sum_{i = 0}^{|V| - 1}\left[\alpha\left(D_i - \sum_{j = 0}^{|V| - 1}f_{ijt} \right) + \beta\sum_{j = 0}^{|V| - 1}\left(f_{ijt}L_{ij}\right) \right] \\\\
    \text{s.t.}& \sum_{j = 0}^{|V| - 1}f_{ijt}\leq I_{jt},&\quad \forall i \in V,\ \forall t \in T \\
    & I_{it} = I_{i(t - 1)} - \sum_{j = 0}^{|V| - 1}f_{ij(t - 1)} + E,&\quad \forall i \in V,\ \forall t \in T \\
    & 0 \leq I_{it} \leq S,&\quad\forall i \in V,\ \forall t \in T \\
    & I_{i0} = S,&\quad\forall i \in V \\
    & 0 \leq f_{ijt},&\quad\forall j \in \left\{v | v \in V, x_v = 1 \right\},\ \forall i \in V,\ \forall t \in T \\
    & 0 = f_{ijt},&\quad\forall j \in \left\{v | v \in V, x_v \neq 1 \right\},\ \forall i \in V,\ \forall t \in T \\
\end{align*}
$$

$$
\begin{align*}
    \argmax_{x}&\ P\left(\sum_{t = 0}^{T - 1}\sum_{i = 0}^{|V| - 1}\sum_{j = 0}^{|V| - 1}f^*_{ijt}\right) - C\sum_{i = 0}^{|V| - 1}x_i\\
    \text{s.t.} &\ x \in \left\{0, 1\right\}^{\left|V\right|}
\end{align*}

$$